# BatchNormalization / LayerNormalization trong tf.keras

Ban minh hoa dung **TensorFlow / Keras** song song voi bai 05 (tu viet `batchnorm_forward/backward`
va `layernorm_forward/backward` bang NumPy). Trong `tf.keras`, ca hai ky thuat nay la lop co san:
`layers.BatchNormalization()` va `layers.LayerNormalization()` - ban chi can **chen dung vi tri**
vao kien truc, khong can tu viet forward/backward hay tu gradient-check.

In [ ]:
# --- Google Colab setup: bo qua tu dong neu chay Jupyter local ---
import sys, os

if "google.colab" in sys.modules:
    REPO_DIR = "/content/AI-Teaching-Labs"
    if not os.path.isdir(REPO_DIR):
        !git clone --depth 1 https://github.com/NonomiyaIzumi/AI-Teaching-Labs.git {REPO_DIR}
    NOTEBOOK_DIR = f"{REPO_DIR}/Deep Learning/Module 1/keras-tensorflow/05_Batch-Norm-Layer-Norm"
    os.chdir(NOTEBOOK_DIR)
    print("Da chuyen working directory sang:", os.getcwd())
else:
    print("Khong phai Colab - dung working directory hien tai:", os.getcwd())
    print("Neu chay Jupyter local: tu thu muc keras-tensorflow/, chay 'uv sync' truoc.")

## 1. Du lieu & kien truc

Dung lai bo du lieu **Pima Indians Diabetes** va kien truc sau `[8, 20, 20, 20, 20, 1]` (4 hidden
layer) giong het bai 05 (NumPy), CO CHU Y khoi tao trong so kem (`stddev=4.0`, tuong duong
`W = randn(...) * 4`) de tao ra tinh huong vanishing/exploding gradient can BatchNorm/LayerNorm
"cuu".

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers

from keras_utils import load_dataset, compile_and_train, evaluate_classification

tf.config.experimental.enable_op_determinism()  # dam bao ket qua giong het nhau giua cac lan chay

train_X, train_Y, test_X, test_Y = load_dataset()
print("train_X:", train_X.shape, " train_Y:", train_Y.shape)

## 2. Chen lop chuan hoa - Exercise 1

O bai 05 (NumPy), ban tu chen `batchnorm_forward`/`layernorm_forward` giua buoc LINEAR va ReLU
trong `forward_propagation`. O day, ban chen `layers.BatchNormalization()`/`layers.LayerNormalization()`
giua `layers.Dense(...)` (khong activation) va `layers.Activation("relu")`.

### Exercise 1 - `build_model_with_norm`

In [ ]:
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0, tuong duong W = randn(...) * 4 o bai 05 NumPy), chen them lop
    chuan hoa (neu co) GIUA moi Dense va ReLU tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong) neu norm_type == "batchnorm": them layers.BatchNormalization()
        #           neu norm_type == "layernorm": them layers.LayerNormalization()
        #           neu norm_type == "none":      khong them gi
        #           sau do LUON them layers.Activation("relu")
        # YOUR CODE STARTS HERE
        pass
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

In [ ]:
#@title Answer { display-mode: "form" }
def build_model_with_norm(input_dim, norm_type="none", bad_init_stddev=4.0):
    '''
    Xay dung mang [input_dim, 20, 20, 20, 20, 1] voi khoi tao trong so CO CHU Y kem
    (RandomNormal stddev=4.0, tuong duong W = randn(...) * 4 o bai 05 NumPy), chen them lop
    chuan hoa (neu co) GIUA moi Dense va ReLU tuy `norm_type`.

    Arguments:
    norm_type -- "none" | "batchnorm" | "layernorm"

    Returns: mot tf.keras.Model CHUA compile.
    '''
    bad_init = keras.initializers.RandomNormal(mean=0.0, stddev=bad_init_stddev, seed=3)
    layer_list = [keras.Input(shape=(input_dim,))]
    for _ in range(4):
        layer_list.append(layers.Dense(20, kernel_initializer=bad_init))
        # (~4 dong)
        # YOUR CODE STARTS HERE
        if norm_type == "batchnorm":
            layer_list.append(layers.BatchNormalization())
        elif norm_type == "layernorm":
            layer_list.append(layers.LayerNormalization())
        layer_list.append(layers.Activation("relu"))
        # YOUR CODE ENDS HERE
    layer_list.append(layers.Dense(1, kernel_initializer=bad_init, activation="sigmoid"))
    model = keras.Sequential(layer_list)
    return model

## 3. Huan luyen & so sanh none / batchnorm / layernorm

Dung `learning_rate=0.1`, full-batch gradient descent (`batch_size=None`) - giong het cau hinh
phan Bonus cua bai 05 (NumPy).

In [ ]:
EPOCHS = 2000
histories = {}
for norm_type in ["none", "batchnorm", "layernorm"]:
    tf.random.set_seed(3)
    model = build_model_with_norm(train_X.shape[0], norm_type=norm_type)
    history = compile_and_train(
        model, train_X, train_Y,
        optimizer=keras.optimizers.SGD(learning_rate=0.1),
        epochs=EPOCHS, batch_size=None, verbose=0,
    )
    histories[norm_type] = {"model": model, "history": history}
    print(f'{norm_type:10s} -> cost cuoi cung (train) = {history.history["loss"][-1]:.4f}')

In [ ]:
plt.figure(figsize=(7, 4.5))
for norm_type in ["none", "batchnorm", "layernorm"]:
    plt.plot(histories[norm_type]["history"].history["loss"], label=norm_type)
plt.xlabel("epoch")
plt.ylabel("cost")
plt.title("Cost qua qua trinh train - so sanh none / batchnorm / layernorm")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
for norm_type in ["none", "batchnorm", "layernorm"]:
    evaluate_classification(histories[norm_type]["model"], test_X, test_Y, f'norm_type = "{norm_type}" (test set)')

## 4. Tong ket

- Ket qua thuc te sau 2000 epoch (`learning_rate=0.1`): **`none`** cost dung yen o **0.6467**
  (~`ln(2)`), test Accuracy 64.9% nhung **Precision = Recall = 0** - dung nhu du doan, khoi tao
  qua lon (`stddev=4.0`) tren mang 4 hidden layer khien mang khong hoc duoc gi. **`batchnorm`** va
  **`layernorm`** deu "cuu" duoc mang: cost train giam manh ve gan 0 (0.0008 va 0.0023), test
  Accuracy len 69.5% (batchnorm, Recall 61.1%) va 64.3% (layernorm, Recall 48.2%).
- Luu y: cost train gan 0 trong khi test accuracy chi ~65-70% la dau hieu **overfitting** ro ret -
  voi 4 hidden layer x 20 unit tren chi 614 vi du train, mang du suc "hoc thuoc" toan bo tap train
  sau 2000 epoch full-batch. Day la mot bai hoc bo sung ngoai pham vi bai 05 goc: BatchNorm/LayerNorm
  giai quyet vanishing/exploding gradient, nhung khong tu dong chong overfitting - can them ky
  thuat khac (regularization, early stopping...) neu muon dong thoi tranh ca hai van de.
- Ve dinh tinh, ket qua nay tai hien dung hien tuong da thay o bai 05 (NumPy): voi khoi tao trong so
  qua lon, mang **`none`** thuong bi ket (cost dung yen o `ln(2)~0.693`, Precision/Recall ve 0) do
  gradient vanishing/exploding; them **BatchNormalization** hoac **LayerNormalization** giup mang
  hoc duoc tro lai.
- Khac biet lon nhat so voi bai 05: ban khong can tu viet `batchnorm_forward`/`backward` (2 Exercise
  gradient-check chinh cua bai 05 NumPy) - `layers.BatchNormalization()`/`layers.LayerNormalization()`
  da duoc kiem chung san boi Keras. Cong viec con lai chi la **biet chen dung cho** (giua Dense va
  activation) va **chon dung loai** (BatchNorm cho CNN/MLP voi batch du lon, LayerNorm cho batch
  nho hoac du lieu tuan tu nhu RNN/Transformer - dung bang so sanh o bai 05 NumPy).
- `layers.BatchNormalization()` tu dong phan biet `training=True/False` (dung thong ke cua batch
  hien tai khi train, dung `moving_mean`/`moving_variance` khi test) - dung chinh xac co che
  `mode="train"`/`mode="test"` ban tu cai trong `batchnorm_forward` o bai 05.